In [31]:
from dataHandler import Config, MetadataHandler
from plots_helpers import *
import pandas as pd
from tabulate import tabulate
from scipy.stats import kruskal, chi2_contingency, mannwhitneyu

import logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

In [185]:
#config_file = "/home/creyna/Vogl-lab_Projects_git/HCC/Metadata/config_survival_trainTest.yaml"
config_file = "/home/creyna/Vogl-lab_Projects_git/HCC/Metadata/config_control_hcc_cirrhosis.yaml"
config = Config(config_file)
metadata_handler = MetadataHandler(config)

In [198]:
clinical_meta = metadata_handler.get_individuals_metadata_df()
clinical_meta['Sex_ctg'] = clinical_meta['Sex'].apply(lambda x: "Male" if x == 1 else "Female" if x == 0 else None)

clinical_meta['Age_ctg'] = clinical_meta['Age'].apply(lambda x: ">=65" if x >=65 else "<65" if x < 65 else None)

clinical_meta['AFP 100_ctg'] = clinical_meta['AFP'].apply(lambda x: ">=100" if x >=100 else "<100" if x < 100 else None)
clinical_meta['AFP 400_ctg'] = clinical_meta['AFP'].apply(lambda x: ">=400" if x >=400 else "<400" if x < 400 else None)

clinical_meta["Line_of_syst.treatment"] = pd.get_dummies(clinical_meta["Line_of_syst.treatment_ctg"], drop_first=True, dtype=int)

clinical_meta['ORR_ctg'] = clinical_meta['ORR_ctg'].apply(lambda x: "yes" if x == "ORR" else "no" if x == "non ORR" else None)
clinical_meta['DCR_ctg'] = clinical_meta['DCR_ctg'].apply(lambda x: "yes" if x == "DCR" else "no" if x == "non DCR" else None)

clinical_meta["BMI Category"] = clinical_meta["BMI"].apply(lambda x: "<18.5" if x < 18.5 else
                                                        ">=18.5<25" if 18.5 <= x < 25 else
                                                        ">=25<30" if 25 <= x < 30 else
                                                        ">=30<35" if 30 <= x < 35 else
                                                        ">=35" if 35 <= x else None)
#clinical_meta = pd.concat([clinical_meta, pd.get_dummies(clinical_meta["BMI Category"], drop_first=False, dtype=int).drop(columns=[">=18.5<25"])], axis =1)

clinical_meta["Child-Pugh_ctg stage"] = clinical_meta["Child-Pugh_ctg stage"].str.replace(r"^stage ", "", regex=True)
clinical_meta["BCLC_ctg stage"] = clinical_meta["BCLC_ctg stage"].str.replace(r"^stage ", "", regex=True)

#clinical_meta = pd.concat([clinical_meta, pd.get_dummies(clinical_meta["Child-Pugh stage_ctg"], drop_first=True, dtype=int).rename(columns={"stage B": "Child-Pugh stage B"})], axis=1)

#clinical_meta = pd.concat([clinical_meta, pd.get_dummies(clinical_meta["BCLC stage_ctg"], drop_first=True, dtype=int).rename(columns={"stage C": "BCLC stage C"})], axis=1)
#clinical_meta = pd.concat([clinical_meta, pd.get_dummies(clinical_meta["mRECIST_ctg"], drop_first=True, dtype=int)], axis = 1)
clinical_meta

,treatment,group_test,Sex,Age,Centre,ORR,ORR_ctg,DCR,DCR_ctg,OS Status,...,AFP,AFP400,AFP400_ctg,mRECIST,mRECIST_ctg,Sex_ctg,Age_ctg,AFP 100_ctg,AFP 400_ctg,BMI Category
SampleName,,,,,,,,,,,,,,,,,,,,,
R14P01_32_HCC3_HCC_MUW_A_T_C2,ICI,HCC,1,54.953425,Vienna,1.0,yes,1.0,yes,1.0,...,404.0,1.0,yes,2.0,PR,Male,<65,>=100,>=400,>=25<30
R14P01_33_HCC25_HCC_MUW_A_T_C2,ICI,HCC,1,71.169863,Vienna,1.0,yes,1.0,yes,1.0,...,63.3,0.0,no,2.0,PR,Male,>=65,<100,<400,>=18.5<25
R14P01_34_HCC27_HCC_MUW_A_T_C2,ICI,HCC,1,64.531507,Vienna,0.0,no,0.0,no,1.0,...,1301.0,1.0,yes,4.0,PD,Male,<65,>=100,>=400,>=18.5<25
R14P01_35_HCC134_HCC_MUW_A_T_C2,ICI,HCC,1,66.438356,Vienna,0.0,no,0.0,no,1.0,...,1348.0,1.0,yes,4.0,PD,Male,>=65,>=100,>=400,>=25<30
R14P01_36_HCC268_HCC_MUW_A_T_C2,ICI,HCC,0,63.997260,Vienna,0.0,no,0.0,no,1.0,...,19727.0,1.0,yes,4.0,PD,Female,<65,>=100,>=400,>=30<35
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
R14P02_46_0175376488_HCC_MUW_A_T_C2,Controls,Controls,0,51.000000,Vienna,NaN,None,NaN,None,NaN,...,NaN,NaN,NaN,NaN,NaN,Female,<65,None,None,None
R14P02_52_0175373320_HCC_MUW_A_T_C2,Controls,Controls,1,75.000000,Vienna,NaN,None,NaN,None,NaN,...,NaN,NaN,NaN,NaN,NaN,Male,>=65,None,None,None
R14P02_58_0175391503_HCC_MUW_A_T_C2,Controls,Controls,1,76.000000,Vienna,NaN,None,NaN,None,NaN,...,NaN,NaN,NaN,NaN,NaN,Male,>=65,None,None,None


In [15]:
# import pickle
# path = "/home/creyna/Vogl-lab_Projects_git/HCC_MUW_analysis/phip_seq_DB/all_libraries_with_info.pkl"
# with open(path, "rb") as f:
#     data = pickle.load(f)
#
# #data['aa_seq'] = data['aa_seq'].str.replace(r' \([^)]*\)$', '', regex=True)
# #data  = data['aa_seq']
# #data.to_csv("/home/creyna/Vogl-lab_Projects_git/HCC_MUW_analysis/phip_seq_DB/aa_seq.csv", sep = ",", index=True)
# path = "/home/creyna/Vogl-lab_Projects_git/HCC_MUW_analysis/phip_seq_DB/all_libraries_with_info.pkl"
# with open(path, "rb") as f:
#     data = pickle.load(f)
#
# data = data[['Organism_complete_name', 'full name','Description', 'pos', 'len_seq', 'aa_seq']]
# data.index = data.index.astype(str)
# ids_df = pd.read_csv("/home/creyna/Vogl-lab_Projects_git/IBD-Chile/ids.txt", header=None, names=["Peptide"], dtype=str)
# peptide_list = ids_df["Peptide"].tolist()
#
# data = data.loc[data.index.isin(peptide_list), :]
# data['aa_seq'] = data['aa_seq'].str.replace(r' \([^)]*\)$', '', regex=True)
# data.to_csv("/home/creyna/Vogl-lab_Projects_git/IBD-Chile/peptide_libraries_info.csv", sep = "\t", index=True)

Index(['nuc_seq', 'pos', 'len_seq', 'full name', 'file', 'aa_seq',
       'end0_len15', 'hash0_len15', 'end1_len15', 'hash1_len15', 'end2_len14',
       'hash2_len14', 'end3_len15', 'hash3_len15', 'end4_len16', 'hash4_len16',
       'is_IEDB_or_cntrl', 'is_pos_cntrl', 'is_neg_cntrl', 'is_rand_cntrl',
       'is_IEDB', 'is_auto', 'is_infect', 'is_EBV', 'is_toxin', 'is_PNP',
       'is_nonPNP_strains', 'is_topgraph', 'is_nontopgraph', 'is_EM', 'is_MPA',
       'is_patho', 'is_IgA', 'is_probio', 'MPA_pass_blast2go',
       'MPA_pass_balst2go_m', 'MPA_Gos', 'MPA_signalp', 'MPA_diamond_toxin',
       'MPA_diamond_flagella', 'diamond_toxin', 'diamond_flagellum', 'signalp',
       'high_abun', 'medium_abun', 'low_abun', 'blast2go', 'blast2go_new',
       'full_topgraph', 'non_PNP_pass_blast2go', 'non_PNP_Gos',
       'non_PNP_pass_balst2go_m', 'non_PNP_signalp', 'non_PNP_diamond_toxin',
       'non_PNP_diamond_flagella', 'IEDB_DOIDs', 'IEDB_comments',
       'IEDB_organism_name', 'IEDB_parent

# Table 1

In [177]:
def summarize_cohort(df, cohort_name,
                     cont_vars,
                     cat_vars_with_levels):
    """
    df            : DataFrame for this cohort
    cohort_name  : e.g. "Controls"
    cont_vars    : [ (col_name, label), ... ]
    cat_vars_with_levels : [ (col_name, label, default_levels), ... ]
         default_levels: list or None. If None, use levels found in df[col].
    """
    N = len(df)
    rows = []
    # Total N row
    rows.append({
        'Var':        'N',
        'Group':      '',
        f'n_{cohort_name}':   N,
        f'%_{cohort_name}':   ''
    })

    # Continuous
    for col, label in cont_vars:
        s = df[col].dropna()
        rows.append({
            'Var':    label,
            'Group':  '',
            f'n_{cohort_name}': f"{s.mean():.1f} ± {s.std():.1f}",
            f'%_{cohort_name}': f"{s.min():.1f}–{s.max():.1f}"
        })

    # Categorical
    for col, label, default_levels in cat_vars_with_levels:
        # decide which levels to iterate
        if default_levels is not None:
            levels = default_levels
        else:
            levels = sorted(df[col].dropna().unique())

        first = True
        for lvl in levels:
            if lvl in df[col].values:
                cnt = (df[col] == lvl).sum()
                pct = cnt / N * 100
                pct_str = f"{pct:.1f}%"
            else:
                # no data in this cohort for that level
                cnt, pct_str = "", ""
            rows.append({
                'Var':    label if first else '',
                'Group':  lvl,
                f'n_{cohort_name}':   cnt,
                f'%_{cohort_name}':   pct_str
            })
            first = False

    return pd.DataFrame(rows)

In [199]:
# Define your specs once:
cont_vars = [('Age', 'Age (years)')]
cat_vars = [
    ('Sex_ctg', 'Sex',          None),
    ('Age_ctg', 'Age', None),
    # For Child-Pugh: Controls has no data, so supply the full list ['A','B','C']
    ('Child-Pugh_ctg stage', 'Child-Pugh stage', ['A','B','C'])]


# Summarize each cohort
controls    = clinical_meta[clinical_meta['group_test']=="Controls"]
hcc         = clinical_meta[clinical_meta['group_test']=="HCC"]
cirrhosis   = clinical_meta[clinical_meta['group_test']=="Cirrhosis"]

df_ctrl = summarize_cohort(controls, "Controls", cont_vars, cat_vars)
df_hcc = summarize_cohort(hcc,       "HCC", cont_vars, cat_vars)
df_cirr = summarize_cohort(cirrhosis, "Cirrhosis", cont_vars, cat_vars)

In [183]:
pvals = []
for col, label in cont_vars:
    # continuous: extract the three groups
    groups = [
        clinical_meta.loc[clinical_meta.group_test=='Controls', col].dropna(),
        clinical_meta.loc[clinical_meta.group_test=='HCC',      col].dropna(),
        clinical_meta.loc[clinical_meta.group_test=='Cirrhosis',col].dropna(),
    ]
    stat, p = kruskal(*groups)
    pvals.append({'Var': label, 'p': f'{p:.2f}'})

for col, label,_ in cat_vars:
    # categorical: build contingency table
    ct = pd.crosstab(
        clinical_meta[col].dropna(),#.fillna('Missing'),
        clinical_meta['group_test']
    )
    stat, p, _, _ = chi2_contingency(ct)

    pvals.append({'Var': label, 'p': f'{p:.2f}'})

pval_df = pd.DataFrame(pvals)
pval_df

,Var,p
0,Age (years),0.60
1,Sex,0.93
2,Age,0.40
3,Child-Pugh stage,0.41


In [165]:
table1_df = df_ctrl.merge(df_hcc.merge(df_cirr, on=['Var', 'Group'], how='left', sort=False), on=['Var', 'Group'], how='right', sort=False)
# Now merge `pval_df` onto  wide summary table `table1_df` on "Var"
table1_df = table1_df.merge(pval_df, on='Var', how='left')

# Round and format
print(tabulate(table1_df, headers='keys', tablefmt='latex', showindex=False))
table1_df

\begin{tabular}{llllllllr}
\hline
 Var              & Group   & n\_Controls   & \%\_Controls   & n\_HCC       & \%\_HCC     & n\_Cirrhosis   & \%\_Cirrhosis   &      p \\
\hline
 N                &         & 72           &              & 108         &           & 77            &               & nan    \\
 Age (years)      &         & 68.8 ± 9.3   & 39.0–86.0    & 67.3 ± 10.2 & 36.8–90.1 & 68.4 ± 8.3    & 39.5–83.7     &   0.6  \\
 Sex              & Female  & 17           & 23.6\%        & 24          & 22.2\%     & 19            & 24.7\%         &   0.93 \\
                  & Male    & 55           & 76.4\%        & 84          & 77.8\%     & 58            & 75.3\%         & nan    \\
 Age              & \ensuremath{<}65     & 21           & 29.2\%        & 42          & 38.9\%     & 26            & 33.8\%         &   0.4  \\
                  & \ensuremath{>}=65    & 51           & 70.8\%        & 66          & 61.1\%     & 51            & 66.2\%         & nan    \\
 Child-Pugh stag

,Var,Group,n_Controls,%_Controls,n_HCC,%_HCC,n_Cirrhosis,%_Cirrhosis,p
0,N,,72,,108,,77,,NaN
1,Age (years),,68.8 ± 9.3,39.0–86.0,67.3 ± 10.2,36.8–90.1,68.4 ± 8.3,39.5–83.7,0.60
2,Sex,Female,17,23.6%,24,22.2%,19,24.7%,0.93
3,,Male,55,76.4%,84,77.8%,58,75.3%,NaN
4,Age,<65,21,29.2%,42,38.9%,26,33.8%,0.40
5,,>=65,51,70.8%,66,61.1%,51,66.2%,NaN
6,Child-Pugh stage,A,NaN,NaN,60,55.6%,49,63.6%,0.41
7,,B,NaN,NaN,41,38.0%,22,28.6%,NaN
8,,C,NaN,NaN,7,6.5%,6,7.8%,NaN


# Table 2

In [249]:
# Define your specs once:
cont_vars = [('Age', 'Age (years)'),
             #('AFP', 'AFP'),
             ]
cat_vars = [
    ('Sex_ctg', 'Sex',          None),
    ('Age_ctg', 'Age', None),
    ('BMI Category', 'BMI', ['>=18.5<25', '>=25<30', '>=30<35', '>=35']),
    # For Child-Pugh: Controls has no data, so supply the full list ['A','B','C']
    ('ORR_ctg', 'ORR', ['yes', 'no']),
    ('DCR_ctg', 'DCR', ['yes', 'no']),
    ('Cirrhosis_ctg', 'Cirrhosis', ['yes', 'no']),
    ('Child-Pugh_ctg stage', 'Child-Pugh stage', ['A','B','C']),
    ('ECOG_ctg', 'ECOG', ['tumor symptoms', 'no tumor symptoms']),
    ('MVI_ctg', 'MVI', ['yes', 'no']),
    ('EHS_ctg', 'EHS', ['yes', 'no']),
    ('BCLC_ctg stage', 'BCLC stage', ['A','B','C', 'D']),
    ('AFP 100_ctg', 'AFP 100', ['>=100', '<100']),
    ('AFP 400_ctg', 'AFP 400', ['>=400', '<400']),
    ('mRECIST_ctg', 'RECIST', ['CR', 'PR', 'SD', 'PD'])
]

# Method 1: simple concatenation
hcc.loc[:, 'treat_centre']= ('HCC_' + hcc['treatment'].astype(str) + '_' + hcc['Centre'].astype(str))
hcc_ici = hcc[(hcc['treatment'] == "ICI")]
hcc_ici_v = hcc[(hcc['treatment'] == "ICI") & (hcc['Centre']    == "Vienna")]
hcc_ici_h    = hcc[(hcc['treatment'] == "ICI") & (hcc['Centre']    == "Hamburg")]
hcc_tki         = hcc[hcc['treatment']=="TKI"]

df_hcc_ici = summarize_cohort(hcc_ici, "ICI", cont_vars, cat_vars)
df_hcc_ici_v = summarize_cohort(hcc_ici_v, "ICI-Vienna", cont_vars, cat_vars)
df_hcc_ici_h = summarize_cohort(hcc_ici_h, "ICI-Hamburg", cont_vars, cat_vars)
df_hcc_tki = summarize_cohort(hcc_tki, "TKI-Vienna", cont_vars, cat_vars)

In [246]:
pvals = []
for col, label in cont_vars:
    # continuous: extract the three groups
    groups = [
        hcc_ici_v[col].dropna(),
        hcc_ici_h[col].dropna(),
        hcc_tki[col].dropna(),
    ]
    stat, p = kruskal(*groups)
    pvals.append({'Var': label, 'p': f'{p:.2f}'})

for col, label,_ in cat_vars:
    # categorical: build contingency table
    ct = pd.crosstab(
        hcc[col].dropna(),#.fillna('Missing'),
        hcc['treat_centre']
    )
    stat, p, _, _ = chi2_contingency(ct)

    pvals.append({'Var': label, 'p': f'{p:.2f}'})

pval_df = pd.DataFrame(pvals)
pval_df

,Var,p
0,Age (years),0.02
1,Sex,0.48
2,Age,0.02
3,BMI,0.30
4,ORR,0.05
5,DCR,0.04
6,Cirrhosis,0.05
7,Child-Pugh stage,0.86
8,ECOG,0.01
9,MVI,0.79


In [277]:
ct

treatment,ICI,TKI
mRECIST_ctg,,
CR,5,4
PD,33,15
PR,18,1
SD,21,10


In [275]:
pvals = []
for col, label in cont_vars:
    # continuous: extract the three groups
    groups = [
        hcc_ici[col].dropna(),
        hcc_tki[col].dropna(),
    ]
    stat, p = mannwhitneyu(*groups, alternative='two-sided')
    pvals.append({'Var': label, 'p': f'{p:.2f}'})

for col, label,_ in cat_vars:
    # categorical: build contingency table
    ct = pd.crosstab(
        hcc[col].dropna(),#.fillna('Missing'),
        hcc['treatment']
    )
    stat, p, _, _ = chi2_contingency(ct, correction=False)

    pvals.append({'Var': label, 'p': f'{p:.2f}'})

pval_df2 = pd.DataFrame(pvals)
pval_df2

,Var,p
0,Age (years),0.01
1,Sex,0.39
2,Age,0.01
3,BMI,1.00
4,ORR,0.16
5,DCR,0.50
6,Cirrhosis,0.85
7,Child-Pugh stage,0.65
8,ECOG,0.28
9,MVI,0.59


In [248]:
# join on index
table2_df = df_hcc_ici_v.join(df_hcc_ici_h.iloc[:,2:4].join(df_hcc_tki.iloc[:,2:4]))
table2_df = table2_df.merge(pval_df, on='Var', how='left')
print(tabulate(table2_df, headers='keys', tablefmt='latex', showindex=False))
table2_df

\begin{tabular}{llllllllr}
\hline
 Var              & Group             & n\_ICI-Vienna   & \%\_ICI-Vienna   & n\_ICI-Hamburg   & \%\_ICI-Hamburg   & n\_TKI-Vienna   & \%\_TKI-Vienna   &      p \\
\hline
 N                &                   & 47             &                & 31              &                 & 30             &                & nan    \\
 Age (years)      &                   & 68.6 ± 8.8     & 38.7–85.9      & 69.7 ± 10.0     & 47.0–86.0       & 62.9 ± 11.4    & 36.8–90.1      &   0.02 \\
 Sex              & Female            & 13             & 27.7\%          & 6               & 19.4\%           & 5              & 16.7\%          &   0.48 \\
                  & Male              & 34             & 72.3\%          & 25              & 80.6\%           & 25             & 83.3\%          & nan    \\
 Age              & \ensuremath{<}65               & 16             & 34.0\%          & 8               & 25.8\%           & 18             & 60.0\%          &   0.02 \\
    

,Var,Group,n_ICI-Vienna,%_ICI-Vienna,n_ICI-Hamburg,%_ICI-Hamburg,n_TKI-Vienna,%_TKI-Vienna,p
0,N,,47,,31,,30,,NaN
1,Age (years),,68.6 ± 8.8,38.7–85.9,69.7 ± 10.0,47.0–86.0,62.9 ± 11.4,36.8–90.1,0.02
2,Sex,Female,13,27.7%,6,19.4%,5,16.7%,0.48
3,,Male,34,72.3%,25,80.6%,25,83.3%,NaN
4,Age,<65,16,34.0%,8,25.8%,18,60.0%,0.02
5,,>=65,31,66.0%,23,74.2%,12,40.0%,NaN
6,BMI,>=18.5<25,20,42.6%,9,29.0%,,,0.30
7,,>=25<30,16,34.0%,14,45.2%,,,NaN
8,,>=30<35,10,21.3%,5,16.1%,,,NaN
9,,>=35,1,2.1%,1,3.2%,,,NaN


In [276]:
# join on index
table3_df = df_hcc_ici.join(df_hcc_tki.iloc[:,2:4])
table3_df = table3_df.merge(pval_df2, on='Var', how='left')
print(tabulate(table3_df, headers='keys', tablefmt='latex', showindex=False))
table3_df

\begin{tabular}{llllllr}
\hline
 Var              & Group             & n\_ICI      & \%\_ICI     & n\_TKI-Vienna   & \%\_TKI-Vienna   &      p \\
\hline
 N                &                   & 78         &           & 30             &                & nan    \\
 Age (years)      &                   & 69.0 ± 9.3 & 38.7–86.0 & 62.9 ± 11.4    & 36.8–90.1      &   0.01 \\
 Sex              & Female            & 19         & 24.4\%     & 5              & 16.7\%          &   0.39 \\
                  & Male              & 59         & 75.6\%     & 25             & 83.3\%          & nan    \\
 Age              & \ensuremath{<}65               & 24         & 30.8\%     & 18             & 60.0\%          &   0.01 \\
                  & \ensuremath{>}=65              & 54         & 69.2\%     & 12             & 40.0\%          & nan    \\
 BMI              & \ensuremath{>}=18.5\ensuremath{<}25         & 29         & 37.2\%     &                &                &   1    \\
                  & \e

,Var,Group,n_ICI,%_ICI,n_TKI-Vienna,%_TKI-Vienna,p
0,N,,78,,30,,NaN
1,Age (years),,69.0 ± 9.3,38.7–86.0,62.9 ± 11.4,36.8–90.1,0.01
2,Sex,Female,19,24.4%,5,16.7%,0.39
3,,Male,59,75.6%,25,83.3%,NaN
4,Age,<65,24,30.8%,18,60.0%,0.01
5,,>=65,54,69.2%,12,40.0%,NaN
6,BMI,>=18.5<25,29,37.2%,,,1.00
7,,>=25<30,30,38.5%,,,NaN
8,,>=30<35,15,19.2%,,,NaN
9,,>=35,2,2.6%,,,NaN
